# SelfAware Dataset — Topic Exploration
Sample 50 questions and use an open-source model to group them into broad topics. We use Phi-3-mini since it is compact and not one of the models we are exploring.

In [8]:
import json
import random

with open('../data/selfaware/SelfAware.json') as f:
    data = json.load(f)

questions = data['example']
print(f"Total questions: {len(questions)}")
print(f"Answerable: {sum(1 for q in questions if q['answerable'])}")
print(f"Unanswerable: {sum(1 for q in questions if not q['answerable'])}")

Total questions: 3369
Answerable: 2337
Unanswerable: 1032


In [9]:
random.seed(42)
sample = random.sample(questions, 50)

print(f"Sampled answerable: {sum(1 for q in sample if q['answerable'])}")
print(f"Sampled unanswerable: {sum(1 for q in sample if not q['answerable'])}")
for q in sample[:5]:
    print(f"  [{q['answerable']}] {q['question']}")

Sampled answerable: 34
Sampled unanswerable: 16
  [False] What came first: the seed or the plant?
  [True] What put an end to the scheme of semi-detached houses?
  [True] When was REM sleep discovered?
  [False] Do all animals have their own souls with their own consciousness or do they share other animals’?
  [True] What is a muslims life purpose?


In [10]:
import torch
from transformers import pipeline

MODEL = "microsoft/Phi-3-mini-4k-instruct"

pipe = pipeline(
    "text-generation",
    model=MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
print("Model loaded.")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Device set to use mps


Model loaded.


In [ ]:
numbered = "\n".join(f"{i+1}. {q['question']}" for i, q in enumerate(sample))

prompt = f"""Below are 50 questions from a QA dataset. Group them into broad topic categories (e.g. Science, History, Geography, Pop Culture, etc.).

For each question, output a single line in this exact format: <number>: <topic>
After all 50 lines, list each unique topic and its count.

Questions:
{numbered}"""

messages = [{"role": "user", "content": prompt}]

result = pipe(messages, max_new_tokens=1024, do_sample=False)
raw_output = result[0]["generated_text"][-1]["content"]
print(raw_output)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/transformers/pytorch_utils.py:335: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48
You are not running the flash-attention implementation, expect numerical differences.


In [ ]:
import re
from collections import Counter

topic_map = {}
for line in raw_output.splitlines():
    m = re.match(r'^(\d+):\s*(.+)$', line.strip())
    if m:
        idx = int(m.group(1)) - 1
        topic = m.group(2).strip()
        if 0 <= idx < len(sample):
            topic_map[idx] = topic

print(f"Parsed topics for {len(topic_map)}/50 questions")
counts = Counter(topic_map.values())
print("\nTopic counts:")
for topic, count in counts.most_common():
    print(f"  {topic}: {count}")

In [ ]:
import matplotlib.pyplot as plt

topics, counts_vals = zip(*counts.most_common())
plt.figure(figsize=(10, 5))
plt.barh(topics[::-1], counts_vals[::-1])
plt.xlabel("Number of questions")
plt.title("SelfAware sample (n=50) — topics assigned by Phi-3-mini")
plt.tight_layout()
plt.show()

In [ ]:
# Drill into a specific topic
target_topic = counts.most_common(1)[0][0]
print(f"Questions labelled '{target_topic}':")
for idx, topic in topic_map.items():
    if topic == target_topic:
        q = sample[idx]
        print(f"  [answerable={q['answerable']}] {q['question']}")